# Memory Room: build a 3D room from your room video

This notebook turns one video of your room into a 3D room made of coloured points, with **MapAnything and its Apache-licensed weights only** (`facebook/map-anything-apache`). It is the first real run of this step. Nothing in it has run on a real video before, so if it fails, the failure is a result to write down, not a reason to start over. The notebook saves a record of the run either way.

**Before you start**
1. Film the room as `docs/filming-guide.md` describes (portrait, 1080p at 30 frames a second, one slow loop).
2. On the laptop, run `python ai/pack_for_colab.py` so the code zip matches the code in the repository.
3. **Runtime → Change runtime type → T4 GPU** (or any GPU) → Save.
4. In Google Drive, make a folder called `MemoryRoom` in *My Drive* and put in it:
   - your **room video** (`.mp4` or `.mov`; if there are several, the newest is used);
   - **`memory-room-ai.zip`**, the code (on the laptop: `ai/dist/memory-room-ai.zip`);
   - optional: a folder **`photos`** holding the photos to place in the room.

Then **Runtime → Run all**. It asks once for access to your Drive. Installing takes about 5 minutes; building the room a few minutes more.

**What you get:** a folder `MyDrive/MemoryRoom/rooms/<run name>/` holding the room (points, cameras, frames, pictures to check) and `run.json` (what ran, how long each step took, how much GPU memory it used, and anything that went wrong), plus a zip of the same folder.

**Privacy:** the frames and the room show the inside of your room. They stay in your Drive and in this Colab session. Photos are copied without their EXIF data, so their GPS position is never kept.

## 1. Settings
The defaults are what the plan calls for.

In [ ]:
DRIVE_FOLDER = "MemoryRoom"    # the folder in My Drive
RUN_NAME = ""                  # empty: made from the date and time, e.g. room-20261118-0930
FRAME_TRIES = (60, 40, 30)     # frames to use; fewer are tried if the GPU runs out of memory
ALIGN_WALLS = False            # True turns the room so its walls run along x and y (not decided yet)
PLACE_PHOTOS = True            # place the photos in MemoryRoom/photos, if there are any
PRECISION = "auto"             # "auto": fp16 on a T4, bf16 on newer GPUs. Use "fp32" if a run stops with
                               # "not real numbers"; it needs about twice the GPU memory

# A rehearsal with a made-up room and no model, used to test this notebook. Leave False.
STAND_IN_MODEL = False
LOCAL_FOLDER = ""              # only for running outside Colab: the folder that stands in for MemoryRoom

## 2. Install MapAnything
The code is pinned to one commit so the run can be repeated. If this cell ends by asking you to restart, do **Runtime → Restart session**, then **Runtime → Run all** again; the install is skipped the second time.

In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

MAPANYTHING_COMMIT = "3d10cf7a3016fc0f9bb13a071ee66c47b10be0d9"   # must match roompipe.model.CODE_COMMIT
IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/map-anything")
MARK = REPO / ".memory-room-installed"


def sh(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        print(done.stdout[-3000:], done.stderr[-3000:])
        raise RuntimeError(f"failed: {' '.join(command)}")
    return done.stdout


if STAND_IN_MODEL:
    print("Rehearsal: nothing is installed.")
elif MARK.exists() and MARK.read_text().strip() == MAPANYTHING_COMMIT:
    print("MapAnything already installed at", MAPANYTHING_COMMIT[:7])
else:
    numpy_before = sys.modules["numpy"].__version__ if "numpy" in sys.modules else None
    if not REPO.exists():
        sh("git", "clone", "-q", "https://github.com/facebookresearch/map-anything.git", str(REPO))
    sh("git", "-C", str(REPO), "fetch", "-q", "origin", MAPANYTHING_COMMIT)
    sh("git", "-C", str(REPO), "checkout", "-q", MAPANYTHING_COMMIT)
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO))
    sh(sys.executable, "-m", "pip", "install", "-q", "pillow-heif")
    MARK.write_text(MAPANYTHING_COMMIT)
    print("MapAnything installed at", MAPANYTHING_COMMIT[:7])
    if numpy_before and importlib.metadata.version("numpy") != numpy_before:
        raise SystemExit("The install changed numpy. Runtime → Restart session, then Runtime → Run all.")

if not STAND_IN_MODEL:
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))  # an editable install is not seen by an already running Python
    # MapAnything pins OpenCV 4.10 over the copy Colab ships. Check it still loads and reads video,
    # in a fresh Python, so a broken install is reported here and not later as "the video could not be opened".
    try:
        print("opencv", sh(sys.executable, "-c", "import cv2; cv2.VideoCapture; print(cv2.__version__)").strip())
    except RuntimeError:
        raise SystemExit("OpenCV does not load after the install. Run in a new cell:  "
                         "!pip install -q --force-reinstall opencv-python-headless==4.10.0.84  "
                         "then Runtime → Restart session, then Runtime → Run all.") from None

## 3. Your Drive, and the code
Unpacks `memory-room-ai.zip` from your Drive folder and loads it.

In [ ]:
import shutil
import zipfile
from datetime import datetime

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    HOME = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK = Path("/content/work")
else:  # a rehearsal on a laptop
    HOME = Path(LOCAL_FOLDER)
    WORK = HOME / "work"
HOME.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

CODE_ZIP = HOME / "memory-room-ai.zip"
if not CODE_ZIP.exists():
    raise SystemExit(f"Put memory-room-ai.zip (from ai/dist/ on the laptop) in {HOME}, then run this cell again.")
CODE = WORK / "code"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    archive.extractall(CODE)
for name in [m for m in sys.modules if m.split(".")[0] in ("roompipe", "roomgeom")]:
    del sys.modules[name]  # so a new zip is picked up when this cell runs again
sys.path.insert(0, str(CODE))

from roomgeom.level import Refilm
from roompipe import model as roompipe_model
from roompipe import session
from roompipe.frames import extract_frames
from roompipe.place import place_photos, summary_lines

PACKED = (CODE / "PACKED.txt").read_text().splitlines()[0]
print("code:", PACKED)
print("weights:", roompipe_model.MODEL_ID, "at revision", roompipe_model.WEIGHTS_REVISION[:7])
assert roompipe_model.CODE_COMMIT == MAPANYTHING_COMMIT, "the notebook and the code zip disagree on the MapAnything commit"

RUN_NAME = RUN_NAME or datetime.now().strftime("room-%Y%m%d-%H%M")
ROOM_DIR = WORK / RUN_NAME
print("this run:", RUN_NAME)

## 4. Check the GPU, and start the run's record
Everything below is written into `run.json`, including failures.

In [ ]:
from IPython.display import Image as Picture
from IPython.display import display

if STAND_IN_MODEL:
    torch = None
    GPU = {"gpu": None, "note": "rehearsal with the stand-in model: no GPU, no reconstruction"}
else:
    import torch

    if not torch.cuda.is_available():
        raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all.")
    card = torch.cuda.get_device_properties(0)
    major, minor = torch.cuda.get_device_capability(0)
    GPU = {"gpu": card.name, "gpu_memory_gb": round(card.total_memory / 1e9, 1),
           "compute_capability": f"{major}.{minor}",
           "precision_used": PRECISION if PRECISION != "auto" else roompipe_model.amp_dtype_for(major),
           # what torch itself claims; on a T4 it may say True although bf16 is only emulated there
           "torch_says_bf16": torch.cuda.is_bf16_supported(),
           "torch": torch.__version__, "cuda": torch.version.cuda}

LOG = session.RunLog(
    peak_memory=torch.cuda.max_memory_allocated if torch else None,
    reset_memory=torch.cuda.reset_peak_memory_stats if torch else None,
    run_name=RUN_NAME,
    stand_in=STAND_IN_MODEL,
    model_id=session.STAND_IN_LABEL if STAND_IN_MODEL else roompipe_model.MODEL_ID,
    weights_revision=None if STAND_IN_MODEL else roompipe_model.WEIGHTS_REVISION,
    mapanything_commit=None if STAND_IN_MODEL else MAPANYTHING_COMMIT,
    code=PACKED,
    settings={"frame_tries": list(FRAME_TRIES), "align_walls": ALIGN_WALLS, "place_photos": PLACE_PHOTOS,
              "precision": PRECISION},
    **GPU,
)
FAILED = None  # set to a sentence when a step fails; later steps then skip, and the record is still saved
print(GPU)

## 5. Find the video and the photos

In [ ]:
videos = session.find_files(HOME, session.VIDEO_ENDINGS)
if not videos:
    raise SystemExit(f"No room video in {HOME}. Upload one (.mp4 or .mov) to that folder, then run this cell again.")
if len(videos) > 1:
    print(f"{len(videos)} videos found; using the newest: {videos[0].name}")
VIDEO = WORK / videos[0].name
if not VIDEO.exists() or VIDEO.stat().st_size != videos[0].stat().st_size:
    shutil.copyfile(videos[0], VIDEO)  # reading from local disk is faster and steadier than from Drive
PHOTOS = session.find_files(HOME / "photos", session.PHOTO_ENDINGS) if PLACE_PHOTOS else []

try:
    facts = session.video_facts(VIDEO)
except Exception as problem:
    facts = {"file": VIDEO.name, "unreadable": f"{type(problem).__name__}: {problem}"}
    FAILED = f"the video {VIDEO.name} could not be opened; save it again as MP4 from the phone's gallery"
    LOG.problem(FAILED)
LOG.note(video=facts, photos_found=len(PHOTOS))
if FAILED:
    print("STOPPED:", FAILED)
else:
    print(f"video: {facts['file']}, {facts['megabytes']} MB, {facts['seconds']} s at {facts['fps']} frames a second")
    if facts["seconds"] and facts["seconds"] < 20:
        print("Note: the video is under 20 seconds. A room usually needs 60 to 90.")
print(f"photos to place: {len(PHOTOS)}")

## 6. Pick the frames
The video is cut into equal stretches, and the sharpest frame of each is kept. Look at the sheet: the frames should be **upright**, sharp and, together, show the whole room.

**If the frames are sideways, stop here.** The levelling step works out "up" from how the frames are held, so sideways frames give a room tipped on its side without any error. Re-save the video from the phone's gallery (or film again) and run again.

In [ ]:
import traceback

FRAMES_DIR = WORK / "frames"
shutil.rmtree(FRAMES_DIR, ignore_errors=True)
FRAMES = []
if FAILED:
    print("skipped:", FAILED)
else:
    try:
        with LOG.step("pick frames"):
            FRAMES = extract_frames(VIDEO, FRAMES_DIR, n_frames=max(FRAME_TRIES))
        LOG.note(frames_picked=len(FRAMES))
        if len(FRAMES) < 30:
            FAILED = f"only {len(FRAMES)} frames could be read from the video; film a longer one"
        else:
            display(Picture(str(session.frames_sheet(FRAMES, WORK / "frames-sheet.png"))))
    except Exception as problem:
        traceback.print_exc()
        FAILED = f"picking frames failed: {type(problem).__name__}: {problem}"
    if FAILED:
        LOG.problem(FAILED)
        print("STOPPED:", FAILED)
    else:
        print(len(FRAMES), "frames picked")

## 7. Load the model
The first time, this downloads about 5 GB of weights.

In [ ]:
import functools
import gc

if FAILED:
    print("skipped:", FAILED)
elif STAND_IN_MODEL:
    stand_in = session.rehearsal_model([p.name for p in FRAMES], [f"{p.stem}.jpg" for p in PHOTOS])
    BUILD_MODEL = LOG.wrap_model(stand_in, "build")
    PLACE_MODEL = LOG.wrap_model(stand_in, "place")
    print("Rehearsal: the stand-in model makes up a room. Nothing below is a reconstruction.")
else:
    try:
        with LOG.step("load model"):
            roompipe_model.load_model()
        BUILD_MODEL = LOG.wrap_model(
            functools.partial(roompipe_model.run_mapanything, keep_views=True, precision=PRECISION), "build")
        PLACE_MODEL = LOG.wrap_model(functools.partial(roompipe_model.run_mapanything, precision=PRECISION), "place")
        print("model loaded; precision:", GPU["precision_used"])
    except Exception as problem:
        traceback.print_exc()
        FAILED = f"loading the model failed: {type(problem).__name__}: {problem}"
        LOG.problem(FAILED)


def free_gpu():
    gc.collect()
    if torch is not None:
        torch.cuda.empty_cache()

## 8. Build the room
One pass of the model over the frames, then: put the floor at height 0, drop the least certain points, and thin the rest to one point per 2 cm. If the GPU runs out of memory, it tries again with fewer frames.

In [ ]:
ROOM = None
if FAILED:
    print("skipped:", FAILED)
else:
    try:
        with LOG.step("build room") as record:
            ROOM, record["attempts"] = session.build_with_fallback(
                FRAMES, BUILD_MODEL, ROOM_DIR, tries=FRAME_TRIES, after_failure=free_gpu, align_walls=ALIGN_WALLS
            )
        LOG.note(room={k: ROOM[k] for k in ("frames_used", "n_points", "has_colors", "box", "up_consistency",
                                            "wall_aligned", "model_info")})
        print(session.room_summary(ROOM))
    except Refilm as problem:
        FAILED = f"the room could not be levelled: {problem}"
    except Exception as problem:
        traceback.print_exc()
        FAILED = f"building the room failed: {type(problem).__name__}: {problem}"
    if FAILED:
        LOG.problem(FAILED)
        print("STOPPED:", FAILED)
    free_gpu()

## 9. Look at it
Does it look like your room? Check that the floor is flat, the walls are straight, and the size is about right; the box size above should be close to the real room. Write down anything that looks wrong: it goes into the write-up as it is.

In [ ]:
LABEL = (session.STAND_IN_LABEL if STAND_IN_MODEL
         else f"{RUN_NAME}: MapAnything, Apache weights, {ROOM['frames_used'] if ROOM else 0} frames")
if ROOM:
    display(Picture(str(session.preview_png(ROOM_DIR, ROOM_DIR / "preview.png", label=LABEL))))
else:
    print("skipped:", FAILED)

## 10. Place the photos (if there are any)
The plan's method: each batch of photos goes through the model twice, with a different half of the stored frames each time. A photo is pinned only if the two answers agree. `ask` means the app would ask you to place it.

In [ ]:
PLACEMENTS = None
if not ROOM:
    print("skipped: there is no room")
elif not PHOTOS:
    print("No photos in MemoryRoom/photos, so nothing to place. Add some and run this cell again.")
else:
    try:
        with LOG.step("prepare photos"):
            prepared = session.prepare_photos(PHOTOS, WORK / "photos-ready")
        LOG.note(photos=prepared)
        for entry in prepared:
            if not entry["file"]:
                print("skipped photo:", entry["photo"], "-", entry["skipped"])
        ready = [WORK / "photos-ready" / e["file"] for e in prepared if e["file"]]
        with LOG.step("place photos"):
            PLACEMENTS = place_photos(ROOM_DIR, ready, PLACE_MODEL)
        print("\n".join(summary_lines(PLACEMENTS)))
        display(Picture(str(session.preview_png(ROOM_DIR, ROOM_DIR / "preview-photos.png", PLACEMENTS,
                                                label=LABEL))))
    except Exception as problem:
        traceback.print_exc()
        LOG.problem(f"placing photos failed: {type(problem).__name__}: {problem}")
    free_gpu()

## 11. Save everything to Drive
This cell runs even after a failure, so the record of what happened is kept.

In [ ]:
ROOM_DIR.mkdir(parents=True, exist_ok=True)
if (WORK / "frames-sheet.png").exists():
    shutil.copyfile(WORK / "frames-sheet.png", ROOM_DIR / "frames-sheet.png")
LOG.note(finished=ROOM is not None, stopped_because=FAILED)
LOG.save(ROOM_DIR)
print(LOG.table())

ZIP = Path(shutil.make_archive(str(WORK / RUN_NAME), "zip", ROOM_DIR))
SAVED = HOME / "rooms" / RUN_NAME
shutil.copytree(ROOM_DIR, SAVED, dirs_exist_ok=True)
shutil.copyfile(ZIP, HOME / "rooms" / ZIP.name)
print(f"\nsaved to {SAVED}\nand {HOME / 'rooms' / ZIP.name} ({ZIP.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(ZIP))

## What next
- Download `MemoryRoom/rooms/<run name>.zip` from Drive to the laptop, **outside** the project folder (for example Downloads), and tell Claude where it is. It shows your room, so it does not go into git.
- To look at the room before the app can: open `points.ply` from the zip in MeshLab or CloudCompare (both free).
- If something failed, keep the zip anyway: `run.json` holds the error, and a documented failed attempt still counts as evidence.